# Notebook de Laboratorio: InAgent Pipeline (Desglosado Paso a Paso)
Este notebook reproduce **el mismo proceso que `src/main.py`**, desglosado celda por celda.
Cada paso está expuesto en variables intermedias (`df_native`, `ctx_raw`, `df_tools_flat`, `df_base`, `df_final`, `df_preparado`, `df_listo`, `df_prod`) para que puedas explorar, filtrar e inspeccionar libremente los `explodes` de `ctx_` y `tool_` sin estar limitado a una función monolítica.

In [1]:
import os
import time
import json
import pyodbc
import logging
import requests
import unicodedata
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
from dotenv import load_dotenv

# Configuración de logging y pandas
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("InAgent_Pipeline")
pd.set_option('display.max_columns', None)

# Cargar variables de entorno
load_dotenv()

API_KEY = os.getenv("INAGENT_API_KEY")
ENDPOINT = os.getenv("INAGENT_URL")

try:
    CREW_MAPPING = json.loads(os.getenv("INAGENT_CREW_MAPPING"))
except (json.JSONDecodeError, TypeError):
    CREW_MAPPING = {
        "766cddc2-eaf9-464e-8f6d-8854ef927ff3": "DENTAL",
        "2539ab63-c408-446a-b13e-1eef4f7c1ba3": "OMV",
        "ad5b9db4-55db-4a2d-b57b-29029c4105f8": "CHECKUP",
        "20ee744a-331f-415c-8d37-68fce098becd": "GRUA",
        "1d140cb8-4714-4840-a4fd-ada8b991e645": "HOGAR"
    }

CREW_IDS = list(CREW_MAPPING.keys())

DB_SERVER = os.getenv('DB_SERVER')
DB_PORT = os.getenv('DB_PORT', '1433')
DB_NAME = os.getenv('BD')
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')

TABLE_NAME = "dbo.inagent"
cdmx_tz = ZoneInfo("America/Mexico_City")

NATIVE_WHITELIST = [
    'Id', 'Id Externo', 'Id Canal', 'Canal', 'Timestamp', 'Inicio', 'Fin', 
    'Duración (s)', 'Análisis Sentimental', 'Tema general de la conversación', 
    'Fue resuelta', 'Fue solo agradecimiento', 'Herramientas Usadas', 
    'Es saliente', 'Fue abandonada', 'Atención del Agente Virtual (s)', 'Contexto', 
    'Resumen', 'Motivo de Cierre', 'Origen'
]

CONTEXT_WHITELIST = [
    'toolLogs', 'tarjeta', 
    'proxyData_sip_attributes_sip_trunkPhoneNumber',
    'proxyData_sip_attributes_sip_h_x-tarjeta-id',
    'comb_resume'
]

TOOL_WHITELIST = [
    'URL_fetch', 'body_fetch', 'return_fetch', 'tool', 'status', 
    'timestamp', 'code_fetch','return_fetch.msg',
    'return_fetch.message','return_fetch.data.client_complete_name',
    'return_fetch.data.policy_number','return_fetch.data.program_name',
    'return_fetch.data.program_status','return_fetch.data.client_account',
    'return_fetch.data.client_telefono','return_fetch.success',
    'return_fetch.data.client_card','body_fetch.cas','body_fetch.cancelMotive',
    'return_fetch.response.cas_folio','return_fetch.cas_folio','body_fetch.scheduleDate',
    'body_fetch.specialty','return_fetch.response.nameDoctor',
    'return_fetch.response.name_service','return_fetch.response.creationDate',
    'return_fetch.response.title','return_fetch.response.provider',
    'return_fetch.response.Kinship','return_fetch.response.category',
    'return_fetch_response.cas_folio',
    'return_fetch.data.data.cas_folio', 'return_fetch.data.data.asistencia_id',
    'return_fetch.data.data.isZonaRoja', 'body_fetch.vehiculo.datos_planos.marca',
    'body_fetch.vehiculo.datos_planos.modelo', 'body_fetch.vehiculo.datos_planos.color',
    'body_fetch.vehiculo.anio', 'body_fetch.vehiculo.placas',
    'body_fetch.sondeo.falla_vehiculo', 'body_fetch.sondeo.falla_detallada',
    'body_fetch.sondeo.tipo_vehiculo', 'body_fetch.sondeo.ubicacion',
    'body_fetch.sondeo.volante_gira', 'body_fetch.ubicacion.origen.referencias',
    'body_fetch.ubicacion.destino.tipo_destino', 'body_fetch.plomeria_sondeo.fuga_visible',
    'body_fetch.plomeria_sondeo.problema_que_presenta', 'body_fetch.electricidad_sondeo.hay_electricidad',
    'body_fetch.electricidad_sondeo.problema_que_presenta', 'body_fetch.cerrajeria_sondeo.problema_que_presenta',
    'body_fetch.vidrieria_sondeo.problema_que_presenta', 'body_fetch.custom_respuesta_agente',
    'body_fetch.phone', 'body_fetch.telefono'
]

PARA_SQL = [
    'Id', 'Id_Externo', 'Id_Canal', 'Canal', 'Timestamp', 'Inicio', 'Fin', 
    'Duracion_s', 'Analisis_Sentimental', 'Tema_general_de_la_conversacion', 
    'Fue_resuelta', 'Fue_solo_agradecimiento', 'Herramientas_Usadas', 
    'Es_saliente', 'Fue_abandonada', 'Origen', 'ctx_tarjeta', 
    'ctx_proxyData_sip_attributes_sip_trunkPhoneNumber', 
    'ctx_proxyData_sip_attributes_sip_h_x_tarjeta_id', 
    'tool_URL_fetch', 'tool_body_fetch', 'tool_return_fetch', 'tool_tool', 
    'tool_status', 'tool_timestamp', 'tool_code_fetch', 'tool_return_fetch_msg', 
    'tool_return_fetch_message', 'tool_return_fetch_data_client_complete_name', 
    'tool_return_fetch_data_policy_number', 'tool_return_fetch_data_program_name', 
    'tool_return_fetch_data_program_status', 'tool_return_fetch_data_client_account', 
    'tool_return_fetch_data_client_telefono', 'tool_return_fetch_success', 
    'tool_return_fetch_data_client_card', 'tool_body_fetch_cas', 
    'tool_body_fetch_cancelMotive', 'tool_return_fetch_response_cas_folio', 
    'tool_body_fetch_scheduleDate', 'tool_body_fetch_specialty', 
    'tool_return_fetch_response_nameDoctor', 'tool_return_fetch_response_name_service', 
    'tool_return_fetch_response_provider', 'tool_return_fetch_response_Kinship', 
    'tool_return_fetch_response_category', 'tool_timestamp_dt', 
    'interaccion_unica', 'Estatus_Final',
    'tool_return_fetch_response_costPIFDoctor', 'tool_return_fetch_response_typeOfService',
    'cas', 'ctx_comb_resume'
]

## 1. Configuración de Fechas y Rango de Extracción

In [2]:
hoy = datetime.now(cdmx_tz)
hace_una_semana = hoy - timedelta(days=7)

start_date = os.getenv("START_DATE") or hace_una_semana.strftime("%Y-%m-%dT%H:%M:%S")
end_date = os.getenv("END_DATE") or hoy.strftime("%Y-%m-%dT%H:%M:%S")

print(f"Extrayendo datos desde {start_date} hasta {end_date}")

Extrayendo datos desde 2026-06-29T00:00:00 hasta 2026-08-18T23:18:38


## 2. Funciones de Extracción API

In [3]:
def to_unix_ms(iso_date):
    dt = datetime.fromisoformat(iso_date.replace("Z", "+00:00"))
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=cdmx_tz)
    return int(dt.timestamp() * 1000)

def safe_json_parse(val):
    try:
        if isinstance(val, (dict, list)): return val
        return json.loads(val) if (val and val != 'null') else {}
    except:
        return {}

def limpiar_nombres(txt):
    if not isinstance(txt, str): return txt
    txt = "".join(c for c in unicodedata.normalize('NFD', txt) if unicodedata.category(c) != 'Mn')
    return txt.replace(" ", "_").replace("(", "").replace(")", "").replace(".", "_")

def extract_inagent_data(start_iso, end_iso):
    all_data = []
    page_size = 100
    max_retries = 5
    cols = []

    logger.info(f"Iniciando extracción desde {start_iso} hasta {end_iso}")

    for crew in CREW_IDS:
        page = 0
        label_origen = CREW_MAPPING.get(crew, "DESCONOCIDO")
        logger.info(f"Extrayendo datos corporativos de {label_origen} (ID: {crew})")
        
        while True:
            params = {
                "crew_id": crew,
                "start_ts": to_unix_ms(start_iso),
                "end_ts": to_unix_ms(end_iso),
                "page": page,
                "pageSize": page_size
            }
            headers = {"apikey": API_KEY}
            
            retry_count = 0
            success = False
            while retry_count < max_retries:
                try:
                    res = requests.get(ENDPOINT, headers=headers, params=params, timeout=30)
                    if res.status_code == 200:
                        success = True
                        break
                    elif res.status_code == 429:
                        retry_count += 1
                        wait_time = 2 ** retry_count
                        logger.warning(f"Too many requests ({label_origen} - Página {page}). Reintentando en {wait_time}s...")
                        time.sleep(wait_time)
                    else:
                        logger.error(f"Error {res.status_code} en {label_origen} pág {page}: {res.text}")
                        break
                except Exception as e:
                    logger.error(f"Error de conexión en {label_origen} pág {page}: {e}")
                    break
                    
            if not success:
                logger.error(f"Agotados reintentos para {label_origen} pág {page}. Se detiene extracción de esta crew.")
                break
                
            payload = res.json().get("data", {})
            rows = payload.get("rows", [])
            
            if page == 0 and crew == CREW_IDS[0]:
                cols = payload.get("dataSchema", {}).get("columnNames", [])
                if "Origen" not in cols:
                    cols.append("Origen")

            if not rows:
                break
            
            for row in rows:
                row.append(label_origen)

            all_data.extend(rows)
            logger.info(f"Página {page} de {label_origen} procesada. Total acumulado: {len(all_data)}")

            if len(rows) < page_size:
                break
            
            page += 1
            time.sleep(0.5) 

    return pd.DataFrame(all_data, columns=cols if cols else None)

## 3. Ejecución de Extracción -> `df_raw`

In [4]:
df_raw = extract_inagent_data(start_date, end_date)
print(f"Filas extraídas en df_raw: {len(df_raw)}")
display(df_raw.head(2))

2026-08-18 23:18:41,410 - INFO - Iniciando extracción desde 2026-06-29T00:00:00 hasta 2026-08-18T23:18:38
2026-08-18 23:18:41,411 - INFO - Extrayendo datos corporativos de DENTAL (ID: 766cddc2-eaf9-464e-8f6d-8854ef927ff3)
2026-08-18 23:18:43,339 - INFO - Página 0 de DENTAL procesada. Total acumulado: 100
2026-08-18 23:18:45,464 - INFO - Página 1 de DENTAL procesada. Total acumulado: 200
2026-08-18 23:18:47,842 - INFO - Página 2 de DENTAL procesada. Total acumulado: 300
2026-08-18 23:18:50,281 - INFO - Página 3 de DENTAL procesada. Total acumulado: 400
2026-08-18 23:18:52,575 - INFO - Página 4 de DENTAL procesada. Total acumulado: 500
2026-08-18 23:18:55,217 - INFO - Página 5 de DENTAL procesada. Total acumulado: 600
2026-08-18 23:18:57,474 - INFO - Página 6 de DENTAL procesada. Total acumulado: 700
2026-08-18 23:18:59,509 - INFO - Página 7 de DENTAL procesada. Total acumulado: 800
2026-08-18 23:19:02,307 - INFO - Página 8 de DENTAL procesada. Total acumulado: 900
2026-08-18 23:19:04,71

Filas extraídas en df_raw: 10681


,Id,Inicio,Fin,Duración (s),Atención del Agente Virtual (s),Atención en Transferencia (s),Análisis Sentimental,Tema general de la conversación,Resumen,Es saliente,Motivo de Cierre,Mensajes del asistente,Mensajes del usuario,Prom. Tpo Respuesta Asistente,Prom. Tpo Respuesta Usuario,Herramientas Usadas,Moderadores Usados,Modificadores Usados,Contexto,Cerrada por inactividad,Cantidad de preguntas a QNA,Fue resuelta,Motivo de abandono,Fue abandonada,Fue transferida,Transferida a,Motivo de transferencia,Fue solo agradecimiento,Id Externo,Id Canal,Canal,Timestamp,Origen
0,07145be6-188f-492e-b6ad-ac401c88b945,2026-08-19T04:31:26.000Z,2026-08-19T04:32:30.090Z,64,64,NaN,neutral,Citas dentales,Usuario reporta error en correos de confirmaci...,False,finished,3.0,1.0,2806,39174,2.0,None,None,"{""TransferenciaAsesorHumano"":{""datos"":{}},""URL...",False,0.0,False,None,False,None,NaN,"Error en confirmación enviada, solicita aclara...",False,,d6aef35f-bbaf-44c9-b960-b31cff41b875,Livekit_SipCall,1787113886000,DENTAL
1,9c2117c9-4ead-42e3-a8fb-79be68d82cf8,2026-08-19T01:47:32.000Z,2026-08-19T01:50:24.493Z,172,167,5.0,neutral,Asistencia dental,Guadalupe pregunta si su familia puede usar su...,False,transfer_completed,5.0,4.0,2150,35833,2.0,None,None,"{""TransferenciaAsesorHumano"":{""datos"":{}},""URL...",False,0.0,False,None,False,True,525571001087,Cancelación de seguro,False,,d6aef35f-bbaf-44c9-b960-b31cff41b875,Livekit_SipCall,1787104052000,DENTAL


## 4. Desglose 1: Selección de Campos Nativos -> `df_native`

In [5]:
df_native = df_raw[[c for c in NATIVE_WHITELIST if c in df_raw.columns]].copy()
print(f"Dimensiones de df_native: {df_native.shape}")
display(df_native.head(2))

Dimensiones de df_native: (10681, 20)


,Id,Id Externo,Id Canal,Canal,Timestamp,Inicio,Fin,Duración (s),Análisis Sentimental,Tema general de la conversación,Fue resuelta,Fue solo agradecimiento,Herramientas Usadas,Es saliente,Fue abandonada,Atención del Agente Virtual (s),Contexto,Resumen,Motivo de Cierre,Origen
0,07145be6-188f-492e-b6ad-ac401c88b945,,d6aef35f-bbaf-44c9-b960-b31cff41b875,Livekit_SipCall,1787113886000,2026-08-19T04:31:26.000Z,2026-08-19T04:32:30.090Z,64,neutral,Citas dentales,False,False,2.0,False,False,64,"{""TransferenciaAsesorHumano"":{""datos"":{}},""URL...",Usuario reporta error en correos de confirmaci...,finished,DENTAL
1,9c2117c9-4ead-42e3-a8fb-79be68d82cf8,,d6aef35f-bbaf-44c9-b960-b31cff41b875,Livekit_SipCall,1787104052000,2026-08-19T01:47:32.000Z,2026-08-19T01:50:24.493Z,172,neutral,Asistencia dental,False,False,2.0,False,False,167,"{""TransferenciaAsesorHumano"":{""datos"":{}},""URL...",Guadalupe pregunta si su familia puede usar su...,transfer_completed,DENTAL


## 5. Desglose 2: Normalización del Contexto -> `ctx_raw` & `ctx_selected`

In [6]:
# Parse de la columna JSON Contexto
ctx_raw = pd.json_normalize(df_raw['Contexto'].apply(safe_json_parse))
ctx_raw.columns = [c.replace(".", "_") for c in ctx_raw.columns]

# Filtrado por CONTEXT_WHITELIST con prefijo ctx_
ctx_selected = ctx_raw[[c for c in CONTEXT_WHITELIST if c in ctx_raw.columns]].add_prefix('ctx_')

# Concatenar nativos + contexto -> df_base
df_base = pd.concat([df_native, ctx_selected], axis=1)
print(f"Dimensiones de df_base (Nativos + Contexto): {df_base.shape}")
display(ctx_selected.head(3))

Dimensiones de df_base (Nativos + Contexto): (10681, 25)


,ctx_toolLogs,ctx_tarjeta,ctx_proxyData_sip_attributes_sip_trunkPhoneNumber,ctx_proxyData_sip_attributes_sip_h_x-tarjeta-id,ctx_comb_resume
0,[{'URL_fetch': 'https://ws-centrodeseguros.tip...,1300001856730003,525599900392,1300001856730003,El cliente RAFAEL REYES ROMERO comenta que sol...
1,[{'URL_fetch': 'https://ws-centrodeseguros.tip...,1300005373218101,525599900392,1300005373218101,La cliente Guadalupe Valenzuela Vega llamó par...
2,[{'URL_fetch': 'https://ws-centrodeseguros.tip...,1300006101775703,525599900392,1300006101775703,NaN


### 🔍 Exploratorio de Campos en `ctx_raw` (Explora libremente nuevas propiedades de Contexto)

In [7]:
print("Todas las columnas disponibles en Contexto desglosado:")
print(list(ctx_raw.columns))
display(ctx_raw.head(3))

Todas las columnas disponibles en Contexto desglosado:
['URL', 'alone_whispered', 'comb_resume', 'creationRequestTimestamp', 'datosSIPObtenidos', 'id-llamada-talk', 'pendingMessages', 'startDelayMilliseconds', 'startedTime', 'tarjeta', 'tarjeta-id', 'toolLogs', 'toolPending', 'turn_sensor_enabled', 'alone_timeout__destroyed', 'alone_timeout__idleNext', 'alone_timeout__idlePrev', 'alone_timeout__idleStart', 'alone_timeout__idleTimeout', 'alone_timeout__onTimeout', 'alone_timeout__repeat', 'proxyData_recordFile', 'proxyData_roomId', 'proxyData_roomName', 'proxyData_sip_attributes_sip_callID', 'proxyData_sip_attributes_sip_callIDFull', 'proxyData_sip_attributes_sip_callStatus', 'proxyData_sip_attributes_sip_callTag', 'proxyData_sip_attributes_sip_h_x-channel-id', 'proxyData_sip_attributes_sip_h_x-id-llamada-talk', 'proxyData_sip_attributes_sip_h_x-lk-call-info', 'proxyData_sip_attributes_sip_h_x-lk-real-ip', 'proxyData_sip_attributes_sip_h_x-lk-transport', 'proxyData_sip_attributes_sip_h_

,URL,alone_whispered,comb_resume,creationRequestTimestamp,datosSIPObtenidos,id-llamada-talk,pendingMessages,startDelayMilliseconds,startedTime,tarjeta,tarjeta-id,toolLogs,toolPending,turn_sensor_enabled,alone_timeout__destroyed,alone_timeout__idleNext,alone_timeout__idlePrev,alone_timeout__idleStart,alone_timeout__idleTimeout,alone_timeout__onTimeout,alone_timeout__repeat,proxyData_recordFile,proxyData_roomId,proxyData_roomName,proxyData_sip_attributes_sip_callID,proxyData_sip_attributes_sip_callIDFull,proxyData_sip_attributes_sip_callStatus,proxyData_sip_attributes_sip_callTag,proxyData_sip_attributes_sip_h_x-channel-id,proxyData_sip_attributes_sip_h_x-id-llamada-talk,proxyData_sip_attributes_sip_h_x-lk-call-info,proxyData_sip_attributes_sip_h_x-lk-real-ip,proxyData_sip_attributes_sip_h_x-lk-transport,proxyData_sip_attributes_sip_h_x-tarjeta-id,proxyData_sip_attributes_sip_hostname,proxyData_sip_attributes_sip_phoneNumber,proxyData_sip_attributes_sip_ruleID,proxyData_sip_attributes_sip_trunkID,proxyData_sip_attributes_sip_trunkPhoneNumber,proxyData_sip_cleanAttributes_callID,proxyData_sip_cleanAttributes_callIDFull,proxyData_sip_cleanAttributes_callStatus,proxyData_sip_cleanAttributes_callTag,proxyData_sip_cleanAttributes_channel-id,proxyData_sip_cleanAttributes_hostname,proxyData_sip_cleanAttributes_id-llamada-talk,proxyData_sip_cleanAttributes_lk-call-info,proxyData_sip_cleanAttributes_lk-real-ip,proxyData_sip_cleanAttributes_lk-transport,proxyData_sip_cleanAttributes_phoneNumber,proxyData_sip_cleanAttributes_ruleID,proxyData_sip_cleanAttributes_tarjeta-id,proxyData_sip_cleanAttributes_trunkID,proxyData_sip_cleanAttributes_trunkPhoneNumber,proxyData_sip_identity,proxyData_sip_kind,proxyData_sip_name,proxyData_stt_language,proxyData_stt_provider,proxyData_stt_version,proxyData_tts_language,proxyData_tts_provider,proxyData_tts_voiceId,alone_timeout__idlePrev_expiry,alone_timeout__idlePrev_id,alone_timeout__idlePrev_msecs,alone_timeout__idlePrev_priorityQueuePosition,citas_agendadas,siniestralidad,API_KEY,fechaHora,alone_timeout__idlePrev__idlePrev__destroyed,alone_timeout__idlePrev__idlePrev__idleStart,alone_timeout__idlePrev__idlePrev__idleTimeout,alone_timeout__idlePrev__idlePrev__repeat,BASE_URL,CRED_SM,cliente_client_account,cliente_client_card,cliente_client_email,cliente_client_lastname,cliente_client_middlename,cliente_client_name,cliente_client_parentesco,cliente_client_phone,cliente_client_phone2,siniestralidad_arrastre_de_grua,CRED_NAME,siniestralidad_cerrajeria,siniestralidad_electricidad,siniestralidad_plomeria,siniestralidad_vidrieria,CRED_SM_TenantId,CRED_SM_active,CRED_SM_createdAt,CRED_SM_createdBy,CRED_SM_description,CRED_SM_id,CRED_SM_name,CRED_SM_privateAttributes_apiKey,CRED_SM_privateAttributesSchema_additionalProperties,CRED_SM_privateAttributesSchema_properties_apiKey_description,CRED_SM_privateAttributesSchema_properties_apiKey_minLength,CRED_SM_privateAttributesSchema_properties_apiKey_type,CRED_SM_privateAttributesSchema_required,CRED_SM_privateAttributesSchema_type,CRED_SM_publicAttributes_headerName,CRED_SM_publicAttributesSchema_additionalProperties,CRED_SM_publicAttributesSchema_properties_headerName_description,CRED_SM_publicAttributesSchema_properties_headerName_minLength,CRED_SM_publicAttributesSchema_properties_headerName_type,CRED_SM_publicAttributesSchema_required,CRED_SM_publicAttributesSchema_type,CRED_SM_type,CRED_SM_updatedAt,CRED_SM_updatedBy
0,https://api.hexalud.com,False,El cliente RAFAEL REYES ROMERO comenta que sol...,2026-08-19T04:31:25.697Z,True,87057ae6ed954064b9517d4cb98c4455,1.0,488.0,2026-08-19T04:31:26.185Z,1300001856730003,1300001856730003,[{'URL_fetch': 'https://ws-centrodeseguros.tip...,False,True,True,NaN,NaN,4225375.0,-1.0,NaN,NaN,d6aef35f-bbaf-44c9-b960-b31cff41b875/2026/08/1...,RM_mYw43dU3Nimk,inagent_522224848891_QtQAreQgAEGp,SCL_R8zfqaJyLo6P,85059fdc-1f5b-4a4c-a580-bae2985e7999,ringing,f7fe04b0-aebc-4628-9ebb-356a024a7b5e,d6aef35f-bbaf-44c9-b960-b31cff41b875,87057a

## 6. Desglose 3: Explode y Normalización de Herramientas (`toolLogs` -> `tool_`)

In [8]:
col_logs = 'ctx_toolLogs'
if col_logs in df_base.columns:
    df_tools_exploded = df_base[['Id', col_logs]].dropna(subset=[col_logs]).explode(col_logs)
    tool_rows = df_tools_exploded[col_logs].apply(lambda x: x if isinstance(x, (dict, list)) else safe_json_parse(x)).tolist()
    df_tools_flat = pd.json_normalize(tool_rows)
    
    cols_t = [c for c in TOOL_WHITELIST if c in df_tools_flat.columns]
    df_tools_final = df_tools_flat[cols_t].copy()
    df_tools_final.columns = [f"tool_{c.replace('.', '_')}" for c in df_tools_final.columns]
    df_tools_final.index = df_tools_exploded.index
    df_tools_merged = pd.concat([df_tools_exploded[['Id']], df_tools_final], axis=1)
    
    df_final = df_base.merge(df_tools_merged, on='Id', how='left')
else:
    df_final = df_base
    
df_final.columns = [c.replace(" ", "_").replace("(", "").replace(")", "").replace(".", "_") for c in df_final.columns]
display(df_final.head(3))

,Id,Id_Externo,Id_Canal,Canal,Timestamp,Inicio,Fin,Duración_s,Análisis_Sentimental,Tema_general_de_la_conversación,Fue_resuelta,Fue_solo_agradecimiento,Herramientas_Usadas,Es_saliente,Fue_abandonada,Atención_del_Agente_Virtual_s,Contexto,Resumen,Motivo_de_Cierre,Origen,ctx_toolLogs,ctx_tarjeta,ctx_proxyData_sip_attributes_sip_trunkPhoneNumber,ctx_proxyData_sip_attributes_sip_h_x-tarjeta-id,ctx_comb_resume,tool_URL_fetch,tool_body_fetch,tool_return_fetch,tool_tool,tool_status,tool_timestamp,tool_code_fetch,tool_return_fetch_message,tool_return_fetch_data_client_complete_name,tool_return_fetch_data_policy_number,tool_return_fetch_data_program_name,tool_return_fetch_data_program_status,tool_return_fetch_data_client_account,tool_return_fetch_data_client_telefono,tool_return_fetch_success,tool_return_fetch_data_client_card,tool_body_fetch_cas,tool_body_fetch_cancelMotive,tool_return_fetch_response_cas_folio,tool_return_fetch_cas_folio,tool_body_fetch_scheduleDate,tool_body_fetch_specialty,tool_return_fetch_response_nameDoctor,tool_return_fetch_response_name_service,tool_return_fetch_response_creationDate,tool_return_fetch_response_title,tool_return_fetch_response_provider,tool_return_fetch_response_Kinship,tool_return_fetch_response_category,tool_return_fetch_data_data_cas_folio,tool_return_fetch_data_data_asistencia_id,tool_return_fetch_data_data_isZonaRoja,tool_body_fetch_vehiculo_datos_planos_marca,tool_body_fetch_vehiculo_datos_planos_modelo,tool_body_fetch_vehiculo_datos_planos_color,tool_body_fetch_vehiculo_anio,tool_body_fetch_vehiculo_placas,tool_body_fetch_sondeo_falla_vehiculo,tool_body_fetch_sondeo_falla_detallada,tool_body_fetch_sondeo_tipo_vehiculo,tool_body_fetch_sondeo_ubicacion,tool_body_fetch_sondeo_volante_gira,tool_body_fetch_ubicacion_origen_referencias,tool_body_fetch_ubicacion_destino_tipo_destino,tool_body_fetch_plomeria_sondeo_fuga_visible,tool_body_fetch_plomeria_sondeo_problema_que_presenta,tool_body_fetch_electricidad_sondeo_hay_electricidad,tool_body_fetch_electricidad_sondeo_problema_que_presenta,tool_body_fetch_cerrajeria_sondeo_problema_que_presenta,tool_body_fetch_vidrieria_sondeo_problema_que_presenta,tool_body_fetch_custom_respuesta_agente,tool_body_fetch_phone,tool_body_fetch_telefono
0,07145be6-188f-492e-b6ad-ac401c88b945,,d6aef35f-bbaf-44c9-b960-b31cff41b875,Livekit_SipCall,1787113886000,2026-08-19T04:31:26.000Z,2026-08-19T04:32:30.090Z,64,neutral,Citas dentales,False,False,2.0,False,False,64,"{""TransferenciaAsesorHumano"":{""datos"":{}},""URL...",Usuario reporta error en correos de confirmaci...,finished,DENTAL,[{'URL_fetch': 'https://ws-centrodeseguros.tip...,1300001856730003,525599900392,1300001856730003,El cliente RAFAEL REYES ROMERO comenta que sol...,https://ws-centrodeseguros.tiprotec.com.mx/api...,NaN,NaN,inicializar_sesion,success,2026-08-19T04:31:30.070Z,200.0,Se encontro la tarjeta correctamente con estat...,RAFAEL REYES ROMERO,1-1I5N64L3,05 PIF PAREJA,Activado por migracion,0000013000018567300,7472689433,NaN,1300001856730003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07145be6-188f-492e-b6ad-ac401c88b945,,d6aef35f-bbaf-44c9-b960-b31cff41b875,Livekit_SipCall,1787113886000,2026-08-19T04:31:26.000Z,2026-08-19T04:32:30.090Z,64,neutral,Citas dentales,False,False,2.0,False,False,64,"{""TransferenciaAsesorHumano"":{""datos"":{}},""URL...",Usuario reporta error en correos de confirmaci...,finished,DENTAL,[{'URL_fetch': 'https://ws-centrodeseguros.tip...,1300001856730003,525599900392,1300001856730003,El cliente RAFAEL REYES ROMERO comenta que sol...,No API call - transferencia SIP,El cliente RAFAEL REYES ROMERO comenta que sol...,El cliente RAFAEL REYES ROMERO comenta que sol...,TransferenciaAsesor,success,2026-08-19T04:32:13.280Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [9]:
df_final.to_csv("raw.csv",index=False)

### 🔍 Exploratorio de `df_tools_flat` (Explora libremente llamadas a tools, payloads y responses)

In [10]:
if not df_tools_flat.empty:
    print("Herramientas más ejecutadas en los logs:")
    print(df_tools_flat['tool'].value_counts() if 'tool' in df_tools_flat.columns else "Sin columna tool")
    print("Columnas disponibles en df_tools_flat:")
    print(list(df_tools_flat.columns))
    display(df_tools_flat.head(5))
else:
    print("No se encontraron registros de herramientas.")

Herramientas más ejecutadas en los logs:
tool
inicializar_sesion                   10646
get_titular                           7312
get_dentistas                         5966
TransferenciaAsesor                   5340
set_checkup                           5062
get_location                          4339
get_servicios_siniestralidad          4183
get_laboratorios                      3910
CerrarSesionPorUsuario                3502
enviar_info_whatsapp                  3392
get_cel_number                        2734
get_fecha                             2239
get_location_coords_fallback          1802
get_beneficiarios                     1542
upsert_beneficiario                   1331
inicializar_sesion_siniestralidad     1151
get_whatsapp_response                  990
enviar_a_whatsapp                      573
crear_servicio_grua                    389
get_horarios                           248
cancelar_cita                          140
CerrarSesionPorTimeout                  82
crear_se

,URL_fetch,body_fetch,code_fetch,status,timestamp,tool,return_fetch.code,return_fetch.data.client_account,return_fetch.data.client_birthday,return_fetch.data.client_birthday_formated,return_fetch.data.client_card,return_fetch.data.client_complete_name,return_fetch.data.client_email,return_fetch.data.client_genre,return_fetch.data.client_lastname,return_fetch.data.client_middlename,return_fetch.data.client_name,return_fetch.data.client_rfc,return_fetch.data.client_telefono,return_fetch.data.client_type,return_fetch.data.effective_date,return_fetch.data.policy_number,return_fetch.data.program_name,return_fetch.data.program_status,return_fetch.data.telefono_10_digitos,return_fetch.httpCode,return_fetch.message,return_fetch.status,return_fetch.xCorrelationId,return_fetch,body_fetch.email,body_fetch.typeOfService,return_fetch.canBookMore,return_fetch.data,return_fetch.servicesAvailable,return_fetch.success,body_fetch.birthDate,body_fetch.fathersLastName,body_fetch.mothersLastName,body_fetch.name,body_fetch.relationship,body_fetch.titularBirthDate,body_fetch.titularEmail,body_fetch.titularName,body_fetch.titularPhone,return_fetch.data.birthDate,return_fetch.data.id,return_fetch.data.name,return_fetch.data.relationship,body_fetch.lat,body_fetch.lng,return_fetch.plus_code.compound_code,return_fetch.plus_code.global_code,return_fetch.results,body_fetch.address,body_fetch.latitude,body_fetch.longitude,return_fetch.confidence,return_fetch.cp,return_fetch.error,return_fetch.formatted_address,return_fetch.source,body_fetch.lat_base,body_fetch.lng_base,return_fetch.response,body_fetch.account,body_fetch.firstDate,body_fetch.firstIdService,body_fetch.firstShift,body_fetch.idBeneficiary,body_fetch.poliza,body_fetch.producto,body_fetch.secondDate,body_fetch.secondIdService,body_fetch.secondShift,body_fetch.thirdDate,body_fetch.thirdShift,return_fetch.cas_folio,return_fetch.idAppointment,body_fetch.custom_respuesta_agente,body_fetch.phone,return_fetch.taskIds,body_fetch.cancelMotive,body_fetch.cas,body_fetch.telefono,return_fetch.idDatAppointment,return_fetch.hs_execution_state,return_fetch.outputFields.atom_error_code,return_fetch.outputFields.atom_error_reason,return_fetch.outputFields.atom_success,body_fetch.respuesta_cliente,body_fetch.scheduleDate,body_fetch.specialty,body_fetch.date,body_fetch.time,return_fetch.response.CatCancelMotive_idCatCancelMotive,return_fetch.response.CatCategories_idCatCategories,return_fetch.response.CatDoctorProfile_idCatDoctorProfile,return_fetch.response.CatExecutiveProfile_idCatExecutiveProfile,return_fetch.response.CatPacienteProfile_idCatPacienteProfile,return_fetch.response.CatServicios_idCatServicios,return_fetch.response.CustomCancelMotive,return_fetch.response.Kinship,return_fetch.response.accessToken,return_fetch.response.bindingIdAppointment,return_fetch.response.cancelDate,return_fetch.response.cas_folio,return_fetch.response.cas_id,return_fetch.response.category,return_fetch.response.codigoPR,return_fetch.response.confirmation_call_appointment_confirmed,return_fetch.response.cost,return_fetch.response.costAppointment,return_fetch.response.costPIF,return_fetch.response.costPIFDoctor,return_fetch.response.createdBy,return_fetch.response.creationDate,return_fetch.response.cuenta,return_fetch.response.dental_subservicio,return_fetch.response.direccion,return_fetch.response.direccion_mad,return_fetch.response.doctorMail,return_fetch.response.duration,return_fetch.response.idAppointment,return_fetch.response.idCatMedicalAnalysisDetails,return_fetch.response.idCatPatientProfile,return_fetch.response.idCatPlatforms,return_fetch.response.idCatSubcampaign,return_fetch.response.idDatAppointment,return_fetch.response.idDoctor,return_fetch.response.idKinship,return_fetch.response.idPatient,return_fetch.response.idRescheduledDatAppointment,return_fetch.response.isInPerson,return_fetch.response.isRescheduled,return_fetch.response.is_call_confirmed,return_fetch.response.meetingId,return_fetch.response.meetingPasswo

In [11]:
df_final

,Id,Id_Externo,Id_Canal,Canal,Timestamp,Inicio,Fin,Duración_s,Análisis_Sentimental,Tema_general_de_la_conversación,Fue_resuelta,Fue_solo_agradecimiento,Herramientas_Usadas,Es_saliente,Fue_abandonada,Atención_del_Agente_Virtual_s,Contexto,Resumen,Motivo_de_Cierre,Origen,ctx_toolLogs,ctx_tarjeta,ctx_proxyData_sip_attributes_sip_trunkPhoneNumber,ctx_proxyData_sip_attributes_sip_h_x-tarjeta-id,ctx_comb_resume,tool_URL_fetch,tool_body_fetch,tool_return_fetch,tool_tool,tool_status,tool_timestamp,tool_code_fetch,tool_return_fetch_message,tool_return_fetch_data_client_complete_name,tool_return_fetch_data_policy_number,tool_return_fetch_data_program_name,tool_return_fetch_data_program_status,tool_return_fetch_data_client_account,tool_return_fetch_data_client_telefono,tool_return_fetch_success,tool_return_fetch_data_client_card,tool_body_fetch_cas,tool_body_fetch_cancelMotive,tool_return_fetch_response_cas_folio,tool_return_fetch_cas_folio,tool_body_fetch_scheduleDate,tool_body_fetch_specialty,tool_return_fetch_response_nameDoctor,tool_return_fetch_response_name_service,tool_return_fetch_response_creationDate,tool_return_fetch_response_title,tool_return_fetch_response_provider,tool_return_fetch_response_Kinship,tool_return_fetch_response_category,tool_return_fetch_data_data_cas_folio,tool_return_fetch_data_data_asistencia_id,tool_return_fetch_data_data_isZonaRoja,tool_body_fetch_vehiculo_datos_planos_marca,tool_body_fetch_vehiculo_datos_planos_modelo,tool_body_fetch_vehiculo_datos_planos_color,tool_body_fetch_vehiculo_anio,tool_body_fetch_vehiculo_placas,tool_body_fetch_sondeo_falla_vehiculo,tool_body_fetch_sondeo_falla_detallada,tool_body_fetch_sondeo_tipo_vehiculo,tool_body_fetch_sondeo_ubicacion,tool_body_fetch_sondeo_volante_gira,tool_body_fetch_ubicacion_origen_referencias,tool_body_fetch_ubicacion_destino_tipo_destino,tool_body_fetch_plomeria_sondeo_fuga_visible,tool_body_fetch_plomeria_sondeo_problema_que_presenta,tool_body_fetch_electricidad_sondeo_hay_electricidad,tool_body_fetch_electricidad_sondeo_problema_que_presenta,tool_body_fetch_cerrajeria_sondeo_problema_que_presenta,tool_body_fetch_vidrieria_sondeo_problema_que_presenta,tool_body_fetch_custom_respuesta_agente,tool_body_fetch_phone,tool_body_fetch_telefono
0,07145be6-188f-492e-b6ad-ac401c88b945,,d6aef35f-bbaf-44c9-b960-b31cff41b875,Livekit_SipCall,1787113886000,2026-08-19T04:31:26.000Z,2026-08-19T04:32:30.090Z,64,neutral,Citas dentales,False,False,2.0,False,False,64,"{""TransferenciaAsesorHumano"":{""datos"":{}},""URL...",Usuario reporta error en correos de confirmaci...,finished,DENTAL,[{'URL_fetch': 'https://ws-centrodeseguros.tip...,1300001856730003,525599900392,1300001856730003,El cliente RAFAEL REYES ROMERO comenta que sol...,https://ws-centrodeseguros.tiprotec.com.mx/api...,NaN,NaN,inicializar_sesion,success,2026-08-19T04:31:30.070Z,200.0,Se encontro la tarjeta correctamente con estat...,RAFAEL REYES ROMERO,1-1I5N64L3,05 PIF PAREJA,Activado por migracion,0000013000018567300,7472689433,NaN,1300001856730003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07145be6-188f-492e-b6ad-ac401c88b945,,d6aef35f-bbaf-44c9-b960-b31cff41b875,Livekit_SipCall,1787113886000,2026-08-19T04:31:26.000Z,2026-08-19T04:32:30.090Z,64,neutral,Citas dentales,False,False,2.0,False,False,64,"{""TransferenciaAsesorHumano"":{""datos"":{}},""URL...",Usuario reporta error en correos de confirmaci...,finished,DENTAL,[{'URL_fetch': 'https://ws-centrodeseguros.tip...,1300001856730003,525599900392,1300001856730003,El cliente RAFAEL REYES ROMERO comenta que sol...,No API call - transferencia SIP,El cliente RAFAEL REYES ROMERO comenta que sol...,El cliente RAFAEL REYES ROMERO comenta que sol...,TransferenciaAsesor,success,2026-08-19T04:32:13.280Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [12]:
df_filt = df_final[df_final['Origen'].astype(str).str.upper().isin(['HOGAR', 'GRUA'])].copy()

In [13]:
col_tool = 'tool_tool' if 'tool_tool' in df_filt.columns else 'tool'

In [14]:
cols_texto = df_filt.select_dtypes(include=['object', 'string']).columns.tolist()
if col_tool in cols_texto: 
    cols_texto.remove(col_tool)
if 'Origen' in cols_texto:
    cols_texto.remove('Origen')

In [15]:
df_largo = df_filt.melt(id_vars=['Origen', col_tool], value_vars=cols_texto, var_name='Columna', value_name='Valor')

In [16]:
df_cas = df_largo[df_largo['Valor'].astype(str).str.contains(r'(?i)CAS\d+', na=False)]

In [17]:
df_resumen = df_cas.groupby(['Origen', col_tool, 'Columna'])['Valor'].agg(
    Total_Casos='count',
    Ejemplos=lambda x: x.unique()[:2].tolist()
).reset_index()

print('--- DESGLOSE DE HERRAMIENTAS Y COLUMNAS QUE GENERAN CAS (HOGAR Y GRUA) ---')
print(df_resumen)

TypeError: unhashable type: 'list'

## 7. Ancla Maestra e Identificador Único Compuesto -> `df_preparado`

In [18]:
def aplicar_ancla_maestra(df):
    df = df.copy()
    df['tool_timestamp_dt'] = pd.to_datetime(
        df['tool_timestamp'], 
        format='mixed', 
        errors='coerce'
    )
    df = df.sort_values(by=['Id', 'tool_timestamp_dt'], ascending=[True, False])
    es_el_ancla = ~df.duplicated(subset=['Id'], keep='first')
    df['interaccion_unica'] = es_el_ancla.astype(np.int8)
    df['Estatus_Final'] = df['tool_tool'].where(es_el_ancla, pd.NA)
    return df

def crear_id_compuesto_pro(df):
    logger.info(" Generando identificadores únicos (id_registro)...")
    df['tool_tool'] = df['tool_tool'].fillna('SIN_HERRAMIENTA')
    tool = df['tool_timestamp'].astype(str)
    df['id_registro'] = (
        df['Id'].astype(str) + "_" + 
        df['tool_tool'].astype(str) + "_" + 
        tool
    )
    return df

df_preparado = aplicar_ancla_maestra(df_final)
df_preparado = crear_id_compuesto_pro(df_preparado)
display(df_preparado[['Id', 'id_registro', 'interaccion_unica', 'Estatus_Final']].head(5))

2026-08-18 23:23:45,213 - INFO -  Generando identificadores únicos (id_registro)...


,Id,id_registro,interaccion_unica,Estatus_Final
374,000085aa-39fa-4d40-9fef-72f649c33d0b,000085aa-39fa-4d40-9fef-72f649c33d0b_Transfere...,1,TransferenciaAsesor
373,000085aa-39fa-4d40-9fef-72f649c33d0b,000085aa-39fa-4d40-9fef-72f649c33d0b_cancelar_...,0,NaN
372,000085aa-39fa-4d40-9fef-72f649c33d0b,000085aa-39fa-4d40-9fef-72f649c33d0b_get_fecha...,0,NaN
371,000085aa-39fa-4d40-9fef-72f649c33d0b,000085aa-39fa-4d40-9fef-72f649c33d0b_inicializ...,0,NaN
45195,00014809-3e82-43be-abbb-ada5fd7a3987,00014809-3e82-43be-abbb-ada5fd7a3987_Transfere...,1,TransferenciaAsesor


## 8. Algoritmo CAS Inteligente Maestro -> `df_listo`

In [19]:
def cas_inteligente_maestro(df_base, col_logs='ctx_toolLogs'):
    df = df_base.copy()

    cols_prioridad = [
        'tool_return_fetch_cas_folio', 
        'tool_body_fetch_cas', 
        'tool_return_fetch_response_cas_folio',
        'tool_return_fetch_data_cas_folio',
        'tool_return_fetch_data_data_cas_folio'
    ]
    
    for col in cols_prioridad:
        if col not in df.columns:
            df[col] = np.nan
        else:
            df[col] = df[col].astype(str).replace(['n.n', 'nan', 'None', 'NaN', 'null', ''], np.nan)

    condiciones_base = [
        df['tool_return_fetch_cas_folio'].astype(str).str.contains('cas', case=False, na=False),
        df['tool_body_fetch_cas'].notna(),
        df['tool_return_fetch_response_cas_folio'].notna(),
        df['tool_return_fetch_data_cas_folio'].notna(),
        df['tool_return_fetch_data_data_cas_folio'].notna(),
        df['tool_return_fetch_cas_folio'].notna()
    ]
    
    valores_base = [
        df['tool_return_fetch_cas_folio'],
        df['tool_body_fetch_cas'],
        df['tool_return_fetch_response_cas_folio'],
        df['tool_return_fetch_data_cas_folio'],
        df['tool_return_fetch_data_data_cas_folio'],
        df['tool_return_fetch_cas_folio']
    ]

    df['cas_fase1'] = np.select(condiciones_base, valores_base, default=np.nan)
    df['cas_fase1'] = df['cas_fase1'].astype(str).str.upper().replace(['NAN', 'NONE', 'NULL', ''], np.nan)

    mask_huerfanos = df['cas_fase1'].isna()
    df['cas_fase2'] = np.nan
    
    if mask_huerfanos.any() and col_logs in df.columns:
        df_rescate = df.loc[mask_huerfanos, ['Id', col_logs]].dropna(subset=[col_logs])
        
        if not df_rescate.empty:
            df_tools_exploded = df_rescate.explode(col_logs)
            
            tool_rows = df_tools_exploded[col_logs].apply(
                lambda x: x if isinstance(x, dict) else safe_json_parse(x)
            ).tolist()
            
            df_tools_flat = pd.json_normalize(tool_rows)
            df_tools_flat.index = df_tools_exploded.index
            df_tools_flat['Id'] = df_tools_exploded['Id']

            rutas_cas = [
                'return_fetch.cas_folio',
                'body_fetch.cas',
                'return_fetch.response.cas_folio',
                'return_fetch.data.cas_folio',
                'return_fetch.data.data.cas_folio'
            ]

            cols_existentes = [c for c in rutas_cas if c in df_tools_flat.columns]
            
            if cols_existentes:
                df_tools_flat['cas_rescatado'] = df_tools_flat[cols_existentes].bfill(axis=1).iloc[:, 0]
                df_tools_flat['cas_rescatado'] = df_tools_flat['cas_rescatado'].astype(str).str.upper().replace(['NAN', 'NONE', 'NULL', ''], np.nan)
                
                df_validos = df_tools_flat.dropna(subset=['cas_rescatado'])
                
                if not df_validos.empty:
                    df_cas_unico = df_validos.drop_duplicates(subset=['Id'], keep='last')[['Id', 'cas_rescatado']]
                    
                    df = df.merge(df_cas_unico, on='Id', how='left')
                    df['cas_fase2'] = df['cas_rescatado']
                    df = df.drop(columns=['cas_rescatado'])

    df['cas_final'] = df['cas_fase1'].fillna(df['cas_fase2'])
    df['cas'] = df['cas_final'].fillna('SIN FOLIO CAS')
    df = df.drop(columns=['cas_fase1', 'cas_fase2', 'cas_final'])

    return df

df_listo = cas_inteligente_maestro(df_preparado)
print("Resumen de folios CAS obtenidos:")
print(df_listo['cas'].value_counts().head(10))

Resumen de folios CAS obtenidos:
cas
SIN FOLIO CAS    25573
CAS200753           36
CAS194824           32
CAS179644           32
CAS206470           31
CAS186914           30
CAS201876           28
CAS182277           27
CAS190795           27
CAS184158           27
Name: count, dtype: int64


## 9. Categorización de Eventos y Mapeos Personalizados

In [20]:
def consolidar_cancel_motive(df):
    df = df.copy()
    cols_cancel_posibles = [
        'tool_body_fetch_cancelMotive',
        'tool_return_fetch_response_CustomCancelMotive',
        'tool_return_fetch_response_CatCancelMotive_idCatCancelMotive',
        'tool_return_fetch_reason',
        'tool_return_fetch_response_title',
        'body_fetch_cancelMotive',
        'return_fetch_response_CustomCancelMotive'
    ]
    otras_cols_cancel = [c for c in df.columns if 'cancelmotive' in c.lower() or 'customcancel' in c.lower()]
    todas_cols_cancel = list(dict.fromkeys(cols_cancel_posibles + otras_cols_cancel))
    cols_existentes = [c for c in todas_cols_cancel if c in df.columns]
    
    if cols_existentes:
        df_temp = df[cols_existentes].copy()
        for c in cols_existentes:
            df_temp[c] = df_temp[c].astype(str).replace(['nan', 'NaN', 'None', 'null', 'n.n', ''], np.nan)
        
        motive_unificado = df_temp.bfill(axis=1).iloc[:, 0]
        df['tool_body_fetch_cancelMotive'] = motive_unificado.fillna('SIN MOTIVO CANCELACION')
    else:
        if 'tool_body_fetch_cancelMotive' not in df.columns:
            df['tool_body_fetch_cancelMotive'] = 'SIN MOTIVO CANCELACION'
            
    return df

def aplicar_taxonomia_eventos(df, columna):
    mapa_regex = {
        r'(?i)transferencia|comunicaci[oó]n asesor': 'Transferencia a Asesor',
        r'(?i)cancelaci[oó]n|cancelar': 'Cancelación',
        r'(?i)reagenda|cambio|correcci[oó]n|apellido error': 'Modificación / Reagenda',
        r'(?i)check[- ]?up|chequeo': 'Consulta Checkup',
        r'(?i)agend|cita|appointment|horario|confirmaci[oó]n': 'Agendamiento / Cita',
        r'(?i)seguro|p[oó]liza|beneficio|privilegio|descuento|programa|peep|pif': 'Seguros y Beneficios',
        r'(?i)validaci[oó]n|verificaci[oó]n|folio|orden|correo|tarjeta|qr|identidad|vigencia': 'Gestión Administrativa',
        r'(?i)dental|m[eé]dic|nutrici[oó]n|psicolog|laboratorio|ambulancia|salud|gr[uú]a|mec[áa]nica': 'Derivación Especialidad / Proveedor',
        r'(?i)asistencia|atenci[oó]n|servicio|orientaci[oó]n': 'Asistencia General'
    }
    condiciones = [df[columna].str.contains(patron, na=False) for patron in mapa_regex.keys()]
    resultados = list(mapa_regex.values())
    df['Evento_Normalizado'] = np.select(condiciones, resultados, default='Sin Clasificar')
    return df

# Categorización de Evento según herramienta principal
condiciones_tool = [
    df_listo['tool_tool'].str.contains(r'(?i)^cerrar', na=False),
    df_listo['tool_tool'].str.contains(r'(?i)^transferencia', na=False),
    df_listo['tool_tool'].str.contains(r'(?i)whatsapp', na=False),
    df_listo['tool_tool'].str.contains(r'(?i)^get_', na=False),
    df_listo['tool_tool'].str.contains(r'(?i)^set_', na=False),
    df_listo['tool_tool'].str.contains(r'(?i)^upsert_', na=False),
    df_listo['tool_tool'].str.contains(r'(?i)cancelar', na=False),
    df_listo['tool_tool'].str.contains(r'(?i)sesi[oó]n', na=False),
    df_listo['tool_tool'].str.contains(r'(?i)herramienta', na=False),
]
resultados_tool = [
    'Cerrar sesion', 
    'Transferencia a Asesor',
    'Interacción WhatsApp',
    'Consulta de Datos',
    "Programar cita",
    "Modificar datos",
    'Cancelación',
    'Gestión de Sesión',
    'Sin evento'
]
df_listo['Evento'] = np.select(condiciones_tool, resultados_tool, default='Sin evento')

# Aplicar taxonomía de regex al tema general
df_listo = aplicar_taxonomia_eventos(df_listo, 'Tema_general_de_la_conversación')

# Consolidación unificada de motivos de cancelación
df_listo = consolidar_cancel_motive(df_listo)

# Mapeos personalizados de columnas para SQL Server
df_listo['Id_Canal'] = df_listo['Origen']
df_listo['Id_Externo'] = df_listo['id_registro'] 
df_listo['ctx_comb_resume'] = df_listo['tool_return_fetch_message']
df_listo['tool_return_fetch_response_nameDoctor'] = df_listo['tool_body_fetch_cas']
df_listo['tool_return_fetch_response_title'] = df_listo['tool_body_fetch_cancelMotive']
df_listo['tool_return_fetch_response_typeOfService'] = df_listo['tool_body_fetch']
df_listo['tool_body_fetch_scheduleDate'] = df_listo['tool_return_fetch_cas_folio']
df_listo['Tema_general_de_la_conversacion'] = df_listo['cas']
df_listo['tool_return_fetch_response_costPIFDoctor'] = df_listo['interaccion_unica']

display(df_listo[['Id', 'Origen', 'Evento', 'Evento_Normalizado', 'cas', 'tool_body_fetch_cancelMotive']].head(5))

,Id,Origen,Evento,Evento_Normalizado,cas,tool_body_fetch_cancelMotive
0,000085aa-39fa-4d40-9fef-72f649c33d0b,DENTAL,Transferencia a Asesor,Derivación Especialidad / Proveedor,HEX205064,SIN MOTIVO CANCELACION
1,000085aa-39fa-4d40-9fef-72f649c33d0b,DENTAL,Cancelación,Derivación Especialidad / Proveedor,HEX205064,"Quiere cambiarla a dental solo, ya no Dental Plus"
2,000085aa-39fa-4d40-9fef-72f649c33d0b,DENTAL,Consulta de Datos,Derivación Especialidad / Proveedor,HEX205064,SIN MOTIVO CANCELACION
3,000085aa-39fa-4d40-9fef-72f649c33d0b,DENTAL,Gestión de Sesión,Derivación Especialidad / Proveedor,HEX205064,SIN MOTIVO CANCELACION
4,00014809-3e82-43be-abbb-ada5fd7a3987,CHECKUP,Transferencia a Asesor,Cancelación,SIN FOLIO CAS,SIN MOTIVO CANCELACION


## 10. Pipeline Final y Limpieza de Tipos -> `df_prod`

In [21]:
def pipeline_maestro_final(df, whitelist):
    df_sql = df.copy()
    
    def limpiar_nombres(txt):
        if not isinstance(txt, str): return txt
        txt = "".join(c for c in unicodedata.normalize('NFD', txt) if unicodedata.category(c) != 'Mn')
        return txt.replace(" ", "_").replace("(", "").replace(")", "").replace(".", "_")

    df_sql.columns = [limpiar_nombres(c) for c in df_sql.columns]
    whitelist_limpia = [limpiar_nombres(c) for c in whitelist]        
    
    if 'cas' not in whitelist_limpia:
        whitelist_limpia.append('cas')

    df_sql = df_sql[[c for c in whitelist_limpia if c in df_sql.columns]]

    cols_num = [
        'Timestamp', 'tool_timestamp', 'Duracion_s', 'Herramientas_Usadas', 'tool_code_fetch',
        'tool_return_fetch_httpCode', 'tool_return_fetch_response_costPIFDoctor',
        'tool_return_fetch_response_selected_dentist','tool_body_fetch_scheduleDate'
    ]
    
    cols_date = ['Inicio', 'Fin', 'tool_timestamp_dt']

    for col in df_sql.columns:
        if col in cols_date:
            df_sql[col] = pd.to_datetime(df_sql[col], errors='coerce')
            df_sql[col] = df_sql[col].astype(object).where(pd.notnull(df_sql[col]), None)
            
        elif col in cols_num:
            df_sql[col] = pd.to_numeric(df_sql[col], errors='coerce')
            if col in ['Timestamp', 'tool_timestamp']:
                df_sql[col] = df_sql[col].fillna(0)
            df_sql[col] = df_sql[col].astype(object).where(pd.notnull(df_sql[col]), None)
            
        elif any(x in col for x in ['Fue_', 'Es_', 'Cerrada_']):
            df_sql[col] = pd.to_numeric(df_sql[col], errors='coerce').fillna(0).astype(int)
            if col in ['Timestamp', 'tool_timestamp']:
                df_sql[col] = df_sql[col].fillna(0)
            
        else:
            df_sql[col] = df_sql[col].astype(str).replace(['n.n','nan', 'None', 'NaN', 'null', ''], None)
            df_sql[col] = df_sql[col].where(df_sql[col].notnull(), None)
            
            if col in ['tool_return_fetch_response_nameDoctor', 'tool_return_fetch_response_name_service', 'tool_return_fetch_data_client_complete_name', 'ctx_proxyData_roomName', 'tool_body_fetch', 'tool_return_fetch', 'ctx_comb_resume', 'tool_URL_fetch']:
                df_sql[col] = df_sql[col].apply(lambda x: x[:255] if isinstance(x, str) else x)
            elif col in ['tool_return_fetch_data_program_name']:
                df_sql[col] = df_sql[col].apply(lambda x: x[:150] if isinstance(x, str) else x)
            elif col in ['Estatus_Final', 'Id_Externo', 'Id', 'Id_Canal', 'ctx_tarjeta', 'tool_tool', 'cas',
            'tool_return_fetch_data_program_status', 'tool_return_fetch_data_client_account',
            'tool_return_fetch_data_client_card',
            'tool_body_fetch_cas', 'tool_return_fetch_data_policy_number',
            'tool_body_fetch_scheduleDate', 'tool_body_fetch_specialty',
            'tool_return_fetch_response_Kinship', 'tool_return_fetch_response_typeOfService',
            'ctx_proxyData_sip_attributes_sip_trunkPhoneNumber', 'ctx_proxyData_sip_attributes_sip_h_x_tarjeta_id',
            'tool_return_fetch_response_cas_folio', 'tool_return_fetch_response_provider', 'tool_return_fetch_response_category']:
                df_sql[col] = df_sql[col].apply(lambda x: x[:100] if isinstance(x, str) else x)
            elif col in ['Origen', 'Canal', 'tool_status', 'tool_return_fetch_status', 'tool_return_fetch_data_client_telefono']:
                df_sql[col] = df_sql[col].apply(lambda x: x[:50] if isinstance(x, str) else x)
            elif col in ['tool_return_fetch_success']:
                df_sql[col] = df_sql[col].apply(lambda x: x[:20] if isinstance(x, str) else x)

    return df_sql

df_sql_preparado = pipeline_maestro_final(df_listo, PARA_SQL)

if 'tool_timestamp' in df_sql_preparado.columns and 'Timestamp' in df_sql_preparado.columns:
    df_sql_preparado['tool_timestamp'] = df_sql_preparado['tool_timestamp'].fillna(df_sql_preparado['Timestamp'])

if 'ctx_proxyData_sip_attributes_sip_h_x-tarjeta-id' in df_sql_preparado.columns:
    df_sql_preparado.rename(columns={
        'ctx_proxyData_sip_attributes_sip_h_x-tarjeta-id': 'ctx_proxyData_sip_attributes_sip_h_x_tarjeta_id'
    }, inplace=True)
    
df_prod = df_sql_preparado[[c for c in PARA_SQL if c in df_sql_preparado.columns]].copy()
df_prod = df_prod.loc[:, ~df_prod.columns.duplicated()].copy()

print(f"Dimensiones de df_prod (Listo para SQL): {df_prod.shape}")
display(df_prod.head(3))

Dimensiones de df_prod (Listo para SQL): (67017, 51)


,Id,Id_Externo,Id_Canal,Canal,Timestamp,Inicio,Fin,Duracion_s,Analisis_Sentimental,Tema_general_de_la_conversacion,Fue_resuelta,Fue_solo_agradecimiento,Herramientas_Usadas,Es_saliente,Fue_abandonada,Origen,ctx_tarjeta,ctx_proxyData_sip_attributes_sip_trunkPhoneNumber,tool_URL_fetch,tool_body_fetch,tool_return_fetch,tool_tool,tool_status,tool_timestamp,tool_code_fetch,tool_return_fetch_message,tool_return_fetch_data_client_complete_name,tool_return_fetch_data_policy_number,tool_return_fetch_data_program_name,tool_return_fetch_data_program_status,tool_return_fetch_data_client_account,tool_return_fetch_data_client_telefono,tool_return_fetch_success,tool_return_fetch_data_client_card,tool_body_fetch_cas,tool_body_fetch_cancelMotive,tool_return_fetch_response_cas_folio,tool_body_fetch_scheduleDate,tool_body_fetch_specialty,tool_return_fetch_response_nameDoctor,tool_return_fetch_response_name_service,tool_return_fetch_response_provider,tool_return_fetch_response_Kinship,tool_return_fetch_response_category,tool_timestamp_dt,interaccion_unica,Estatus_Final,tool_return_fetch_response_costPIFDoctor,tool_return_fetch_response_typeOfService,cas,ctx_comb_resume
0,000085aa-39fa-4d40-9fef-72f649c33d0b,000085aa-39fa-4d40-9fef-72f649c33d0b_Transfere...,DENTAL,Livekit_SipCall,1787080028000,2026-08-18 19:07:08+00:00,2026-08-18 19:12:38.008000+00:00,330,neutral,Asistencia dental,0,0,4.0,0,0,DENTAL,1300004966400104,525599900392,No API call - transferencia SIP,La titular MARIA DE JESUS AGUILAR GARCIA solic...,La titular MARIA DE JESUS AGUILAR GARCIA solic...,TransferenciaAsesor,success,0.0,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SIN MOTIVO CANCELACION,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,2026-08-18 19:12:20.350000+00:00,1,TransferenciaAsesor,1,La titular MARIA DE JESUS AGUILAR GARCIA solic...,HEX205064,NaN
1,000085aa-39fa-4d40-9fef-72f649c33d0b,000085aa-39fa-4d40-9fef-72f649c33d0b_cancelar_...,DENTAL,Livekit_SipCall,1787080028000,2026-08-18 19:07:08+00:00,2026-08-18 19:12:38.008000+00:00,330,neutral,Asistencia dental,0,0,4.0,0,0,DENTAL,1300004966400104,525599900392,https://api.hexalud.com/api/pif2.0/dental/cancel,NaN,NaN,cancelar_cita,error,0.0,400.0,No se permite cancelar la cita porque está pro...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,HEX205064,"Quiere cambiarla a dental solo, ya no Dental Plus",NaN,None,NaN,HEX205064,NaN,NaN,NaN,NaN,2026-08-18 19:10:11.392000+00:00,0,NaN,0,NaN,HEX205064,No se permite cancelar la cita porque está pro...
2,000085aa-39fa-4d40-9fef-72f649c33d0b,000085aa-39fa-4d40-9fef-72f649c33d0b_get_fecha...,DENTAL,Livekit_SipCall,1787080028000,2026-08-18 19:07:08+00:00,2026-08-18 19:12:38.008000+00:00,330,neutral,Asistencia dental,0,0,4.0,0,0,DENTAL,1300004966400104,525599900392,No API call,NaN,8/18/2026 1:10:09 p.m.,get_fecha,success,0.0,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SIN MOTIVO CANCELACION,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,2026-08-18 19:10:09.147000+00:00,0,NaN,0,NaN,HEX205064,NaN


## 11. Función Envolvente `transform_data()` (Referencia Directa 1:1 con `main.py`)
Esta función empaqueta exactamente todos los pasos desglosados anteriormente (4 al 10) en una sola llamada. Puedes usarla cuando requieras procesar en bloque o copiarla a `main.py`.

In [22]:
def transform_data(df_raw):
    if df_raw.empty: return pd.DataFrame()
    logger.info("Iniciando transformaciones de datos...")

    df_native = df_raw[[c for c in NATIVE_WHITELIST if c in df_raw.columns]].copy()
    
    ctx_raw = pd.json_normalize(df_raw['Contexto'].apply(safe_json_parse))
    ctx_raw.columns = [c.replace(".", "_") for c in ctx_raw.columns]
    
    ctx_selected = ctx_raw[[c for c in CONTEXT_WHITELIST if c in ctx_raw.columns]].add_prefix('ctx_')
    
    df_base = pd.concat([df_native, ctx_selected], axis=1)

    col_logs = 'ctx_toolLogs'
    if col_logs in df_base.columns:
        df_tools_exploded = df_base[['Id', col_logs]].dropna(subset=[col_logs]).explode(col_logs)
        tool_rows = df_tools_exploded[col_logs].apply(lambda x: x if isinstance(x, (dict, list)) else safe_json_parse(x)).tolist()
        df_tools_flat = pd.json_normalize(tool_rows)
        
        cols_t = [c for c in TOOL_WHITELIST if c in df_tools_flat.columns]
        df_tools_final = df_tools_flat[cols_t].copy()
        df_tools_final.columns = [f"tool_{c.replace('.', '_')}" for c in df_tools_final.columns]
        df_tools_final.index = df_tools_exploded.index
        df_tools_merged = pd.concat([df_tools_exploded[['Id']], df_tools_final], axis=1)
        
        df_final = df_base.merge(df_tools_merged, on='Id', how='left')
    else:
        df_final = df_base
        
    df_final.columns = [c.replace(" ", "_").replace("(", "").replace(")", "").replace(".", "_") for c in df_final.columns]
    
    df_preparado = aplicar_ancla_maestra(df_final)
    df_preparado = crear_id_compuesto_pro(df_preparado)
    
    # CAS inteligente maestro
    df_listo = cas_inteligente_maestro(df_preparado, col_logs='ctx_toolLogs')
    
    # Evento categorization
    condiciones = [
        df_listo['tool_tool'].str.contains(r'(?i)^cerrar', na=False),
        df_listo['tool_tool'].str.contains(r'(?i)^transferencia', na=False),
        df_listo['tool_tool'].str.contains(r'(?i)whatsapp', na=False),
        df_listo['tool_tool'].str.contains(r'(?i)^get_', na=False),
        df_listo['tool_tool'].str.contains(r'(?i)^set_', na=False),
        df_listo['tool_tool'].str.contains(r'(?i)^upsert_', na=False),
        df_listo['tool_tool'].str.contains(r'(?i)cancelar', na=False),
        df_listo['tool_tool'].str.contains(r'(?i)sesi[oó]n', na=False),
        df_listo['tool_tool'].str.contains(r'(?i)herramienta', na=False),
    ]

    resultados = [
        'Cerrar sesion', 
        'Transferencia a Asesor',
        'Interacción WhatsApp',
        'Consulta de Datos',
        "Programar cita",
        "Modificar datos",
        'Cancelación',
        'Gestión de Sesión',
        'Sin evento'
    ]

    df_listo['Evento'] = np.select(condiciones, resultados, default='Sin evento')

    df_listo = aplicar_taxonomia_eventos(df_listo, 'Tema_general_de_la_conversación')
    
    # Consolidación unificada de motivos de cancelación
    df_listo = consolidar_cancel_motive(df_listo)
    
    # Mapeo personalizado / Asignaciones
    df_listo['Id_Canal'] = df_listo['Origen']
    df_listo['Id_Externo'] = df_listo['id_registro'] 
    df_listo['ctx_comb_resume'] = df_listo['tool_return_fetch_message']
    df_listo['tool_return_fetch_response_nameDoctor'] = df_listo['tool_body_fetch_cas']
    df_listo['tool_return_fetch_response_title'] = df_listo['tool_body_fetch_cancelMotive']
    df_listo['tool_return_fetch_response_typeOfService'] = df_listo['tool_body_fetch']
    df_listo['tool_body_fetch_scheduleDate'] = df_listo['tool_return_fetch_cas_folio']
    df_listo['Tema_general_de_la_conversacion'] = df_listo['cas']
    df_listo['tool_return_fetch_response_costPIFDoctor'] = df_listo['interaccion_unica']

    df_listo = pipeline_maestro_final(df_listo, PARA_SQL)

    if 'tool_timestamp' in df_listo.columns and 'Timestamp' in df_listo.columns:
        df_listo['tool_timestamp'] = df_listo['tool_timestamp'].fillna(df_listo['Timestamp'])
    
    if 'ctx_proxyData_sip_attributes_sip_h_x-tarjeta-id' in df_listo.columns:
        df_listo.rename(columns={
            'ctx_proxyData_sip_attributes_sip_h_x-tarjeta-id': 'ctx_proxyData_sip_attributes_sip_h_x_tarjeta_id'
        }, inplace=True)
        
    df_produccion = df_listo[[c for c in PARA_SQL if c in df_listo.columns]].copy()
    df_produccion = df_produccion.loc[:, ~df_produccion.columns.duplicated()].copy()
    
    return df_produccion

## 12. Función de Carga a SQL Server (`load_to_sql`)

In [23]:
def load_to_sql(df_para_sql):
    if df_para_sql.empty:
        logger.info("No hay datos extraídos de la API para procesar.")
        return

    conn_str = f"DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={DB_SERVER},{DB_PORT};DATABASE={DB_NAME};UID={DB_USER};PWD={DB_PASS};TrustServerCertificate=yes"
    
    try:
        conn = pyodbc.connect(conn_str)
        cursor = conn.cursor()
        cursor.fast_executemany = True

        df_para_sql = df_para_sql.loc[:, ~df_para_sql.columns.duplicated()].copy()
        if 'id_registro' in df_para_sql.columns:
            df_para_sql = df_para_sql.drop(columns=['id_registro'])

        df_para_sql = df_para_sql.drop_duplicates(subset=['Id_Externo']).copy()
        
        cols_to_trim_255 = ['tool_return_fetch_response_nameDoctor', 'tool_return_fetch_response_name_service', 
                            'tool_return_fetch_data_client_complete_name', 'ctx_proxyData_roomName', 
                            'tool_body_fetch', 'tool_return_fetch', 'ctx_comb_resume', 'tool_URL_fetch']
        for c in cols_to_trim_255:
            if c in df_para_sql.columns:
                df_para_sql[c] = df_para_sql[c].astype(str).str.slice(0, 255)
                df_para_sql[c] = df_para_sql[c].replace(['None', 'nan', 'NaN', 'null'], None)

        cols = df_para_sql.columns.tolist()
        col_names_bracketed = ", ".join(f"[{c}]" for c in cols)
        
        cursor.execute(f"IF OBJECT_ID('tempdb..#stg_inagent') IS NOT NULL DROP TABLE #stg_inagent")
        cursor.execute(f"SELECT TOP 0 {col_names_bracketed} INTO #stg_inagent FROM {TABLE_NAME}") 

        placeholders = ", ".join("?" for _ in cols)
        sql_insert = f"INSERT INTO #stg_inagent ({col_names_bracketed}) VALUES ({placeholders})"

        sql_types = {}
        try:
            cursor.execute(f"""
                SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH, NUMERIC_PRECISION, NUMERIC_SCALE
                FROM INFORMATION_SCHEMA.COLUMNS
                WHERE TABLE_NAME = '{TABLE_NAME.replace("dbo.", "")}'
            """)
            sql_types = {row[0]: {'type': row[1], 'char_len': row[2], 'prec': row[3], 'scale': row[4]} for row in cursor.fetchall()}

            for col in cols:
                if col in sql_types:
                    sql_type = sql_types[col]['type'].lower()
                    if sql_type in ('int', 'smallint', 'tinyint', 'bigint'):
                        for idx, row in enumerate(df_para_sql.values):
                            val = row[cols.index(col)]
                            if val is not None and isinstance(val, str) and val.isdigit():
                                val_int = int(val)
                                if sql_type == 'int' and val_int > 2147483647:
                                    logger.warning(f"COLUMNA '{col}' VALOR FUERA DE RANGO INT en fila {idx}: {val}")
                                elif sql_type == 'bigint' and val_int > 9223372036854775807:
                                    logger.warning(f"COLUMNA '{col}' VALOR FUERA DE RANGO BIGINT en fila {idx}: {val}")
        except Exception as type_e:
            logger.warning(f"No se pudo auditar tipos SQL: {type_e}")

        inputsizes = []
        for col in cols:
            db_type_info = sql_types.get(col, {})
            db_type = db_type_info.get('type', '').lower()
            char_len = db_type_info.get('char_len', None)
            
            if 'char' in db_type or 'text' in db_type:
                non_null_strings = df_para_sql[col].dropna().astype(str)
                max_str_len = non_null_strings.str.len().max() if not non_null_strings.empty else 0
                if char_len is None or char_len == -1:
                    bind_len = max(max_str_len, 4000)
                else:
                    bind_len = max(max_str_len, char_len)
                inputsizes.append((pyodbc.SQL_WVARCHAR, bind_len, 0))
            else:
                inputsizes.append(None)
                
        cursor.setinputsizes(inputsizes)

        def clean_sql_value(v):
            if pd.isna(v):
                return None
            if isinstance(v, np.integer):
                return int(v)
            if isinstance(v, np.floating):
                return float(v) if not np.isnan(v) else None
            return v

        data_to_load = [tuple(clean_sql_value(v) for v in row) for row in df_para_sql.values]

        logger.info(f"Subiendo {len(data_to_load)} registros a Staging DB...")

        try:
            cursor.executemany(sql_insert, data_to_load)
        except Exception as e:
            logger.error(f"Fallo masivo en executemany. Inyectando auditoría de aislamiento... Error original: {e}")

            for i, row in enumerate(data_to_load):
                try:
                    cursor.execute(sql_insert, row)
                except Exception as inner_e:
                    logger.error(f"FILA CORRUPTA AISLADA EN EL ÍNDICE {i}")
                    for j, (columna, valor) in enumerate(zip(cols, row)):
                        sql_type = sql_types.get(columna, {}).get('type', 'UNKNOWN')
                        logger.error(f"  [{j}] Columna: {columna} | SQL_Type: {sql_type} | Valor: {valor} | Tipo_Python: {type(valor)}")
                        try:
                            single_sql = f"INSERT INTO #stg_inagent ([{columna}]) VALUES (?)"
                            cursor.execute(single_sql, (valor,))
                            cursor.execute("DELETE FROM #stg_inagent WHERE 1=0")
                        except Exception as col_e:
                            logger.error(f"  >>>>> COLUMNA CULPABLE IDENTIFICADA: {columna} | Valor: {valor} | Error: {col_e}")
                    break

            raise

        sql_merge = f"""
        MERGE {TABLE_NAME} AS target
        USING #stg_inagent AS source
        ON (target.Id_Externo = source.Id_Externo)
        WHEN MATCHED THEN
            UPDATE SET 
                target.Analisis_Sentimental = source.Analisis_Sentimental,
                target.Tema_general_de_la_conversacion = source.Tema_general_de_la_conversacion,
                target.tool_status = source.tool_status,
                target.tool_return_fetch_message = source.tool_return_fetch_message,
                target.cas = source.cas,
                target.ctx_comb_resume = source.ctx_comb_resume
        WHEN NOT MATCHED THEN
            INSERT ({col_names_bracketed})
            VALUES ({", ".join(f"source.[{c}]" for c in cols)});
        """
        
        logger.info("Ejecutando MERGE en tabla definitiva...")
        cursor.execute(sql_merge)
        conn.commit()
        logger.info(f"ÉXITO: {len(df_para_sql)} registros sincronizados correctamente en {TABLE_NAME}.")

    except Exception as e:
        if 'conn' in locals(): conn.rollback()
        logger.error(f"Error en SQL: {e}")
    finally:
        if 'cursor' in locals(): cursor.close()
        if 'conn' in locals(): conn.close()

## 13. Ejecución de Carga a SQL Server

In [24]:
if 'df_prod' in locals() and not df_prod.empty:
    load_to_sql(df_prod)
else:
    print("df_prod no está listo o está vacío. Omitiendo carga.")

2026-08-18 23:25:30,766 - INFO - Subiendo 66881 registros a Staging DB...
2026-08-18 23:27:20,084 - INFO - Ejecutando MERGE en tabla definitiva...
2026-08-18 23:27:30,824 - INFO - ÉXITO: 66881 registros sincronizados correctamente en dbo.inagent.
